``SequenceFeature().acc()`` builds an **order-aware scale baseline** for a prediction model. For each sequence it measures, per scale and lag ``k``, the mean-centered auto-covariance of that scale's values along a span, giving the ``(n_seq, n_scales * n_lag)`` matrix ``X``. It is the lag extension of :class:`SequenceFeature` ``scale_composition``: where the scale composition reduces each scale to one span mean (order-blind), the auto-covariance keeps short-range sequential order in scale-space while staying position-agnostic. Its purpose is **comparison**: it is the natural middle point between the order-blind scale mean and a :class:`CPP` ``feature_matrix`` (fully positional). Here we load the ``DOM_GSEC`` example dataset and the bundled scales (see [Breimann25]_):

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False
df_seq = aa.load_dataset(name="DOM_GSEC")
df_scales = aa.load_scales()
sf = aa.SequenceFeature()

By default (``list_parts=None``, ``n_lag=1``) the whole TMD-JMD span (``jmd_n`` + ``tmd`` + ``jmd_c``) is used and one lag-1 value per scale is returned. Residues that are not in the scale index (gaps ``'-'`` and other non-canonical symbols) are dropped before the covariance is taken:

In [2]:
X = sf.acc(df_seq=df_seq, df_scales=df_scales)
print(f"n samples: {X.shape[0]}")
print(f"n scales:  {df_scales.shape[1]}")
print(f"Shape of X: {X.shape}")

n samples: 126
n scales:  586
Shape of X: (126, 586)


Use ``n_lag`` to add more lags. Lags ``1 .. n_lag`` are computed and the output width becomes ``n_scales * n_lag``, in lag-major order (all scales for lag 1, then all scales for lag 2, ...). Larger lags need longer spans (lag ``k`` needs at least ``k + 1`` scored residues):

In [3]:
X_lag3 = sf.acc(df_seq=df_seq, df_scales=df_scales, n_lag=3)
print(f"Shape of X (n_lag=3): {X_lag3.shape}")

Shape of X (n_lag=3): (126, 1758)


With ``return_df=True`` the matrix is returned as a labeled ``pd.DataFrame`` (rows indexed by protein entry, one column per scale-and-lag named ``f"{scale}_lag{k}"``):

In [4]:
df_acc = sf.acc(df_seq=df_seq, df_scales=df_scales, n_lag=2, return_df=True)
aa.display_df(df_acc, n_rows=10, show_shape=True)

DataFrame shape: (126, 1172)


Use ``list_parts`` to cover only selected sequence parts (concatenated per sequence). For example, restrict the baseline to the TMD:

In [5]:
X_tmd = sf.acc(df_seq=df_seq, df_scales=df_scales, list_parts="tmd", n_lag=2)
print(f"Shape of X (TMD only, n_lag=2): {X_tmd.shape}")

Shape of X (TMD only, n_lag=2): (126, 1172)


**Further parameters.** ``df_scales`` defaults to the bundled scales (or ``options['df_scales']``) when omitted, so ``sf.acc(df_seq=df_seq)`` returns the lag-1 auto-covariance over the default scale set:

In [6]:
X_default = sf.acc(df_seq=df_seq)
print(f"Shape of X (default scales): {X_default.shape}")

Shape of X (default scales): (126, 586)
